In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# USAGE
# ==============================================================================
print("""
USAGE
-----

This interactive notebook demonstrates the decomposition of a non-minimum-phase
discrete-time LTI system into a minimum-phase system and an all-pass system.

The slider controls the parameter c, with |c| < 1. The original zero is located
at z0 = 1/c*, while the corresponding minimum-phase zero is located at zmin = c.

Moving the slider therefore changes the position of the original zero and its
corresponding minimum-phase zero.

The notebook displays four plots in a 2x2 grid:

1. Original system:
   Pole-zero diagram of the non-minimum-phase system.

2. Minimum-phase system:
   Pole-zero diagram after mapping the zero according to z -> 1/z*.

3. Magnitude response:
   Comparison of the magnitude responses of the original and minimum-phase
   systems. They should be identical.

4. Phase response:
   Comparison of the phase responses. The all-pass component changes the phase
   while leaving the magnitude unchanged.

The notebook also verifies numerically that the all-pass filter has unit
magnitude, |H_ap(e^(jω))| = 1.

The example is based on the decomposition

    H(z) = H_min(z) H_ap(z).

Move the slider and observe how the zero moves between the original and
minimum-phase representations while the magnitude response remains unchanged.
""")

# ==============================================================================
# Interactive parameter
# ==============================================================================
c_slider = widgets.FloatSlider(
    value=-1/3, min=-0.8, max=-0.2, step=0.01,
    description='c:', continuous_update=True
)
out = widgets.Output()

# ==============================================================================
# Main plotting function
# ==============================================================================
def plot_minimum_phase(c):
    with out:
        clear_output(wait=True)

        # ----------------------------------------------------------------------
        # System definition
        #
        # H(z) = H_min(z) H_ap(z)
        #
        # Original zero:
        #       z0 = 1/c*
        #
        # Minimum-phase zero:
        #       zmin = c
        #
        # Pole:
        #       p = -1/2
        # ----------------------------------------------------------------------
        z0 = 1/np.conj(c)
        zmin = c
        pole = -0.5

        # ----------------------------------------------------------------------
        # Transfer functions evaluated on the unit circle
        # ----------------------------------------------------------------------
        w = np.linspace(0, np.pi, 1000)
        ejw = np.exp(1j*w)

        H_original = (1 - z0*ejw**-1) * (-3/z0) / (1 - pole*ejw**-1)
        H_min = (1 - zmin*ejw**-1) * (-3/z0) / (1 - pole*ejw**-1)
        H_ap = H_original / H_min

        # ----------------------------------------------------------------------
        # Numerical magnitude and phase
        # ----------------------------------------------------------------------
        mag_original = np.abs(H_original)
        mag_min = np.abs(H_min)
        phase_original = np.unwrap(np.angle(H_original))
        phase_min = np.unwrap(np.angle(H_min))
        mag_ap = np.abs(H_ap)

        # ----------------------------------------------------------------------
        # Figure: 2x2 grid
        # ----------------------------------------------------------------------
        fig, ax = plt.subplots(
            2, 2,
            figsize=(9, 6.2),
            constrained_layout=False
        )

        fig.subplots_adjust(
    left=0.08, right=0.97,
    top=0.88, bottom=0.13,
    wspace=0.28, hspace=0.80
)

        # ----------------------------------------------------------------------
        # Plot 1: Original pole-zero diagram
        # ----------------------------------------------------------------------
        a = ax[0, 0]
        theta = np.linspace(0, 2*np.pi, 400)

        a.plot(
            np.cos(theta), np.sin(theta),
            'k--', linewidth=1
        )

        a.scatter(
            np.real(z0), np.imag(z0),
            s=90, marker='o',
            facecolors='none',
            edgecolors='C0',
            linewidths=2,
            label='Zero'
        )

        a.scatter(
            np.real(pole), np.imag(pole),
            s=90, marker='x',
            color='C3',
            linewidths=2,
            label='Pole'
        )

        a.axhline(0, color='black', linewidth=0.7)
        a.axvline(0, color='black', linewidth=0.7)

        a.set_xlim(-5.5, 1.5)
        a.set_ylim(-2, 2)
        a.set_aspect('equal')

        a.set_title('Original System')
        a.set_xlabel('Re{z}')
        a.set_ylabel('Im{z}')
        a.grid(alpha=0.25)

        # Legend moved below the corresponding plot
        a.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.28),
            ncol=2,
            fontsize=8,
            frameon=True
        )

        # ----------------------------------------------------------------------
        # Plot 2: Minimum-phase pole-zero diagram
        # ----------------------------------------------------------------------
        a = ax[0, 1]

        a.plot(
            np.cos(theta), np.sin(theta),
            'k--', linewidth=1
        )

        a.scatter(
            np.real(zmin), np.imag(zmin),
            s=90, marker='o',
            facecolors='none',
            edgecolors='C2',
            linewidths=2,
            label='Minimum-phase zero'
        )

        a.scatter(
            np.real(pole), np.imag(pole),
            s=90, marker='x',
            color='C3',
            linewidths=2,
            label='Pole'
        )

        a.axhline(0, color='black', linewidth=0.7)
        a.axvline(0, color='black', linewidth=0.7)

        a.set_xlim(-1.5, 1.5)
        a.set_ylim(-1.5, 1.5)
        a.set_aspect('equal')

        a.set_title('Minimum-Phase System')
        a.set_xlabel('Re{z}')
        a.set_ylabel('Im{z}')
        a.grid(alpha=0.25)

        a.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.28),
            ncol=2,
            fontsize=8,
            frameon=True
        )

        # ----------------------------------------------------------------------
        # Plot 3: Magnitude responses
        # ----------------------------------------------------------------------
        a = ax[1, 0]

        a.plot(
            w, mag_original,
            linewidth=1.8,
            label='Original'
        )

        a.plot(
            w, mag_min,
            '--',
            linewidth=1.8,
            label='Minimum phase'
        )

        a.set_title('Magnitude Response')
        a.set_xlabel('ω')
        a.set_ylabel('|H(eʲω)|')
        a.set_xlim(0, np.pi)
        a.grid(alpha=0.25)

        a.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.28),
            ncol=2,
            fontsize=8,
            frameon=True
        )

        # ----------------------------------------------------------------------
        # Plot 4: Phase responses
        # ----------------------------------------------------------------------
        a = ax[1, 1]

        a.plot(
            w, phase_original,
            linewidth=1.8,
            label='Original'
        )

        a.plot(
            w, phase_min,
            '--',
            linewidth=1.8,
            label='Minimum phase'
        )

        a.set_title('Phase Response')
        a.set_xlabel('ω')
        a.set_ylabel('Phase (rad)')
        a.set_xlim(0, np.pi)
        a.grid(alpha=0.25)

        a.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.28),
            ncol=2,
            fontsize=8,
            frameon=True
        )

        # ----------------------------------------------------------------------
        # Overall title
        # ----------------------------------------------------------------------
        fig.suptitle(
            f'Minimum-Phase Decomposition   '
            f'(c = {c:.2f}, z₀ = {z0.real:.3f}, z_min = {zmin.real:.3f})',
            fontsize=13
        )

        plt.show()

        # ----------------------------------------------------------------------
        # Numerical verification
        # ----------------------------------------------------------------------
        print(f"Original zero       : {z0.real:.4f}")
        print(f"Minimum-phase zero  : {zmin.real:.4f}")
        print(f"Pole                : {pole:.4f}")
        print(f"|original zero|     : {abs(z0):.4f}")
        print(f"|minimum zero|      : {abs(zmin):.4f}")
        print(f"|H_ap(eʲω)| ≈       : {np.mean(mag_ap):.4f}")


# ==============================================================================
# Interactive output
# ==============================================================================
display(c_slider)
display(out)

widgets.interactive_output(
    plot_minimum_phase,
    {'c': c_slider}
)

# ==============================================================================
# Initial display
# ==============================================================================
plot_minimum_phase(c_slider.value)